⚠️ **Under construction.** <!-- banner:under-construction --> This lab is drafted but not yet instructor-reviewed; it may change without notice until its session.

# Lab 9 · Reading week: two levels deep

**Today:** two reading drills on nested draws, then PS3.

**Before you start:** labs 7 and 8. Short on purpose.

Each section names one idea, explains what it does, and asks you to **predict
what a cell prints before you run it**. Write the prediction down, on paper, out
loud, or in a comment. A prediction you can compare against the output is what
tells you which parts of the code you can already read.

Most sections end with a **Test your understanding** task: write a small piece
of code, then run the check cell under it. The check never grades and never
breaks anything. A ⬜ means not attempted yet, a ❌ means not yet and comes with
a hint, and a ✅ means passing. Run the check cells rather than editing them.
Everything else in the notebook is yours to change.

**AI in this lab.** Until your prediction is written down, work at level 1, with
no AI. The prediction is how you find out what you can read unaided, and both
exams are level 1. Once you have run a cell, level 3 is encouraged: ask your
tutor to explain anything you missed.

Run every cell, and change things to see what happens. Nothing in this notebook
can be broken in a way that matters.

## 1 · Which line is the hidden layer?

**Before running: this loop has three random draws in it. Which one is the hidden layer — the setting the outside world never sees — and how many times does each line run?**

In [ ]:
import numpy as np

rng = np.random.default_rng(73)
totals = []
for _ in range(4):
    efficiency = rng.beta(8, 2)          # line A
    total = 0
    for _ in range(3):
        made = rng.poisson(50)           # line B
        kept = rng.binomial(made, efficiency)   # line C
        total += kept
    totals.append(total)
print(totals)

Line A runs 4 times — once per outer pass — and is the hidden layer: each batch draws its own efficiency, and all three inner runs share it. Lines B and C run 12 times each. Reading a nested simulation *is* this: for each random line, which loop owns it, and therefore what stays fixed while what varies.

**Test your understanding.** Rewrite the world so the efficiency is drawn **inside** the inner loop (a fresh efficiency per run, none shared). Wrap it as `independent_total(rng)` returning one batch's total, and check it against the forced case: with `rng.beta(8, 2)` replaced by a certain efficiency of... no — keep beta(8, 2); the checks below use seeds.

In [ ]:
# your turn: independent_total(rng) — efficiency drawn per run, 3 runs of poisson(50)
import numpy as np

In [ ]:
# run, don't edit — self-check
from labcheck import check

check("independent_total", expect=140, args=(np.random.default_rng(79),),
      hint="per run: efficiency = rng.beta(8, 2), made = rng.poisson(50), kept = rng.binomial(made, efficiency) — in that order, three times")
check("independent_total", expect=115, args=(np.random.default_rng(83),))

## 2 · Spot the planted mismatch

A colleague claims this chunk simulates "2,000 days, each day's rate drawn fresh, then one count per day." **Read it against the claim before running. One line makes the claim false — which?**

In [ ]:
import numpy as np

rng = np.random.default_rng(89)
rate = rng.gamma(5, 4)
counts = []
for _ in range(2000):
    counts.append(rng.poisson(rate))
print("mean", round(np.mean(counts), 1), " sd", round(np.std(counts), 1))

The `rate` line sits **outside** the loop: drawn once, shared by all 2,000 days — one hidden setting, not 2,000 fresh ones. The printed sd is the tell (near √rate, one-level noise). Indentation is not decoration; it is the claim.

**Test your understanding.** Fix it to match the claim: `fresh_rate_sd(n_days, rng)` — rate drawn inside the loop each day (gamma(5, 4)), one poisson count per day; return the counts' sd rounded to 1 decimal. Predict first: bigger or smaller than the printed one?

In [ ]:
# your turn: fresh_rate_sd(n_days, rng)
import numpy as np

In [ ]:
# run, don't edit — self-check
from labcheck import check

check("fresh_rate_sd", expect=10.1, args=(3000, np.random.default_rng(97)),
      hint="rate inside the loop; gamma(5, 4) then poisson(rate), each day")

## If you finish early

PS3 — the design section closes Friday.

## If you are stuck

Wave someone over — this hour exists so a stuck step costs you a minute rather
than an evening. Known snags:

- **`independent_total` misses the seed checks** — draw order inside each run must be exactly: beta, poisson, binomial. Any other order pulls different numbers off the stream.
- **Section 2's fix prints nearly the same sd as the broken version on your seed** — compare against the forced reasoning instead: a shared rate adds no day-to-day drift; a fresh rate must widen the spread.